In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, precision_score, f1_score, recall_score
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import make_pipeline
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.pipeline import make_pipeline as imb_make_pipeline
import numpy as np
from sklearn.preprocessing import PowerTransformer
from sklearn.model_selection import RandomizedSearchCV




In [ ]:
files = [
    "/home/truphile/Downloads/classification data/data_descriptions.csv",
    "/home/truphile/Downloads/classification data/train.csv",
    "/home/truphile/Downloads/classification data/test.csv"
]

datasets = [pd.read_csv(file) for file in files]

data, train, test = datasets
data.head()

In [ ]:
train.head()

In [ ]:
train["Churn"].value_counts()

In [ ]:
train["Churn"].value_counts(normalize=True) * 100 #check percentage for value_count

In [ ]:
train.isnull()
train.drop_duplicates()

In [ ]:
correlation = train.corr(numeric_only=True)
correlation["Churn"].sort_values(ascending=False)  # corelation check (0.5 - 1 = shows there is a strong positive correlation),(0.1 - 0.4 = shows there is a weak positive correlation)
                                                   # corelation check (-0.5 - 1 = shows there is a strong negative correlation),(-0.1 - -0.4 = shows there is a weak negative correlation)

In [ ]:
train.drop(columns="CustomerID")

In [ ]:
X = train.drop(columns='Churn')
y = train["Churn"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify=train["Churn"],random_state=42)

In [ ]:
num_column = make_pipeline(
    PowerTransformer(method='yeo-johnson', standardize=True))
cat_column = make_pipeline(
        OneHotEncoder(handle_unknown='ignore'))

processed = ColumnTransformer([
        ( 'num', num_column, make_column_selector(dtype_include=np.number)),
        ('cat', cat_column, make_column_selector(dtype_exclude=np.number))
    ])

In [ ]:
brfc = make_pipeline(processed, BalancedRandomForestClassifier(random_state=42, n_jobs=-1))

In [ ]:
y_pred_brfc = cross_val_predict(brfc, X_train, y_train, cv=5, method='predict_proba', n_jobs=-1)[:, 1]
threshold = 0.65
y_pred_custom = (y_pred_brfc >= threshold).astype(int)
print(confusion_matrix(y_train, y_pred_custom))
print(classification_report(y_train,y_pred_custom))

print("Precision:", precision_score(y_train, y_pred_custom))
print("Recall:", recall_score(y_train, y_pred_custom))
print("F1:", f1_score(y_train, y_pred_custom))

In [ ]:
X_train_processed = processed.fit_transform(X_train)

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300, 500],

    'max_depth': [5, 10, 15, 20, None],

    'min_samples_split': [2, 5, 10, 20, 50],

    'min_samples_leaf': [1, 2, 5, 10, 20],

    'max_features': ['sqrt', 'log2', 0.5],

    'sampling_strategy': [0.3, 0.5, 0.7, 1.0]
}

search = RandomizedSearchCV(
    estimator=brfc,
    param_distributions=param_grid,
    n_iter=30,
    scoring='f1',
    cv=5,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)
print(search.best_params_)